In [1]:
# Import required libraries.
import pandas as pd
from IPython.display import display  # noqa: A004

# Dataset

The same dataset, FoodAPS, and the same preparation procedures as the previous notebook are used for this notebook.


In [2]:
hh_df = pd.read_csv("data/faps_household_puf.csv")

item_df = pd.read_csv("data/faps_fahitem_puf.csv")
nutrient_df = pd.read_csv("data/faps_fahnutrients.csv")

ID_VARIABLES = ["hhnum", "eventid", "itemnum"]
REQUIRED_VARIABLES = [*ID_VARIABLES, "totgramsedible", "totsug"]

food_df = item_df.merge(nutrient_df[REQUIRED_VARIABLES], how="left", on=ID_VARIABLES)

# Data Preprocessing


The same preprocessing procedures were made, but compressed into a single code block.


In [3]:
import enum
import warnings

hh_df = hh_df[(hh_df["numguests"] == 0) & (hh_df["hhsizechange"] == 0)]


class SNAPNowAdmin(enum.IntEnum):
    """Results from match of household members with SNAP administrative data."""

    NO_MATCH = 0
    """No match."""
    CONFIRMED_SNAP = 1
    """Match confirms SNAP participation."""
    CONFIRMED_NON_SNAP = 2
    """Match confirms SNAP nonparticipation."""
    VALID_SKIP = -996
    """Valid skip. (Consent for data matching not given.)"""


hh_df = hh_df[
    (hh_df["snapnowadmin"] == SNAPNowAdmin.CONFIRMED_SNAP) | (hh_df["snapnowadmin"] == SNAPNowAdmin.CONFIRMED_NON_SNAP)
]

with warnings.catch_warnings():
    # Ignore the `PerformanceWarning`
    warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

    hh_df["snap_status"] = (hh_df["snapnowadmin"] == SNAPNowAdmin.CONFIRMED_SNAP).astype("int8")

hh_df["region"] = hh_df["region"].map({1: "NORTHEAST", 2: "MIDWEST", 3: "SOUTH", 4: "WEST"})

food_df = food_df[(food_df["totgramsedible"] > 0) & (food_df["totsug"].notna())]

The same convenience variables were calculated, but were filtered to only include what's needed, `sugar_amount` and
`high_sugar_status`, and compressed into a single code block.


In [4]:
food_df["sugar_amount"] = (food_df["totsug"] / 100) * food_df["totgramsedible"]

HIGH_SUGAR_THRESHOLD = 22.5

food_df["high_sugar_status"] = (food_df["totsug"] > HIGH_SUGAR_THRESHOLD).astype("int8")

The household sugar data frame, calculated for the exploratory data analysis questions, was also lifted over into this
notebook.


In [5]:
hh_sugar_df = (
    food_df.groupby("hhnum")
    .agg(total_sugar_amount=("sugar_amount", "sum"), total_amount=("totgramsedible", "sum"))
    .reset_index()
)
hh_sugar_df["sugar_share"] = hh_sugar_df["total_sugar_amount"] / hh_sugar_df["total_amount"]

# Add the household number and the SNAP status back for identification and categorization.
hh_sugar_df = hh_sugar_df.merge(hh_df[["hhnum", "snap_status", "region"]], how="left", on="hhnum")

# Data Mining

The rule mining technique was used to find patterns and relationships between household characteristics and sugar
purchasing behavior. It is well-suited for identifying associations between categorical variables and expressing them as
simple, interpretable "if–then" rules.


The food dataset was merged with selected household variables (`snap_status` and `region`) using the household number
(`hhnum`) as the key. A left join was used to retain all food records.


In [6]:
merged_df = food_df.merge(hh_df[["hhnum", "snap_status", "region"]], how="left", on="hhnum")

The data was grouped by household number (`hhnum`) to create a household-level dataset. For each household,
`high_sugar_status` was set to `1` if any transaction indicated high sugar consumption, while `snap_status` and `region`
were taken as the first recorded values.

The result is a summarized dataset with only one row per household.


In [7]:
hh_transactions = (
    merged_df.groupby("hhnum")
    .agg({"high_sugar_status": lambda s: int((s == 1).any()), "snap_status": "first", "region": "first"})
    .reset_index()
)

A copy of the household-level dataset was created and transformed into a format suitable for rule mining.

Binary indicator columns were generated for SNAP participation (`SNAP`) and high sugar status (`HIGH_SUGAR`), and
`region` was converted into dummy variables. Unnecessary columns were then removed.


In [8]:
basket = hh_transactions.copy()
basket["SNAP"] = (basket["snap_status"] == 1).astype("int8")
basket["HIGH_SUGAR"] = basket["high_sugar_status"]

region_dummies = pd.get_dummies(basket["region"], prefix="REGION", dtype="int8")

basket = pd.concat([basket, region_dummies], axis=1)
basket = basket.drop(columns=["hhnum", "snap_status", "region", "high_sugar_status"])

Frequent item sets were generated from the dataset using the Apriori algorithm, with a minimum support threshold of
`0.05`. This identifies combinations of variables that appear together in at least 5% of the households.


In [9]:
from mlxtend.frequent_patterns import apriori

with warnings.catch_warnings():
    # Ignore the `DeprecationWarning`
    warnings.filterwarnings("ignore", category=DeprecationWarning)

    frequent_itemsets = apriori(basket, min_support=0.05, use_colnames=True)

Association rules were generated from the frequent item sets using lift as the evaluation metric, with a minimum
threshold of `1`.

The resulting rules were then filtered to identify those where SNAP participation appears in the antecedent and high
sugar status appears in the consequent.


In [10]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules[(rules["antecedents"].apply(lambda x: "SNAP" in x)) & (rules["consequents"].apply(lambda x: "HIGH_SUGAR" in x))]

rules[["antecedents", "consequents", "support", "confidence", "lift"]]

,antecedents,consequents,support,confidence,lift
0,frozenset({SNAP}),frozenset({REGION_SOUTH}),0.105785,0.443564,3.772078
1,frozenset({REGION_SOUTH}),frozenset({SNAP}),0.105785,0.899598,3.772078
2,frozenset({REGION_WEST}),frozenset({SNAP}),0.058087,0.911111,3.820352
3,frozenset({SNAP}),frozenset({REGION_WEST}),0.058087,0.243564,3.820352
4,frozenset({HIGH_SUGAR}),frozenset({REGION_SOUTH}),0.083589,0.119112,1.012928
5,frozenset({REGION_SOUTH}),frozenset({HIGH_SUGAR}),0.083589,0.710843,1.012928
6,"frozenset({HIGH_SUGAR, SNAP})",frozenset({REGION_SOUTH}),0.075325,0.451841,3.842466
7,"frozenset({HIGH_SUGAR, REGION_SOUTH})",frozenset({SNAP}),0.075325,0.901130,3.778500
8,"frozenset({SNAP, REGION_SOUTH})",frozenset({HIGH_SUGAR}),0.075325,0.712054,1.014652
9,frozenset({HIGH_SUGAR}),"frozenset({SNAP, REGION_SOUTH})",0.075325,0.107335,1.014652


## Analysis

The association rule results show strong relationships between region and SNAP participation, but weak relationships
involving high sugar status. Rules like `{REGION_SOUTH} → {SNAP}` and `{SNAP} → {REGION_WEST}` have relatively high
confidence (about `0.44`–`0.90`) and lift around `3.77`, meaning these combinations occur more often than expected.

In contrast, rules involving high sugar status have lift values close to `1.01`, indicating little to no meaningful
association with region or SNAP participation.


# Statistical Inference


## Independent t-Test


The mean of the sugar shares of participating and nonparticipating households was compared using an independent
two-sample t-test, assuming variances between the groups are not equal.


In [11]:
from scipy.stats import ttest_ind

snap = hh_sugar_df[hh_sugar_df["snap_status"] == 1]["sugar_share"]
non_snap = hh_sugar_df[hh_sugar_df["snap_status"] == 0]["sugar_share"]

t_stat, p_val = ttest_ind(snap, non_snap, equal_var=False)

print(f"t-statistic = {t_stat:4f}")
print(f"p-value     = {p_val:4f}")

t-statistic = -0.300006
p-value     = 0.764680


### Results

The independent t-test gave `t = -0.3001` and `p = 0.7647`. Since the p-value is greater than `α = 0.05`, there is
insufficient evidence to reject the null hypothesis..

This means there is **no significant difference** between the two group means. Any observed difference is likely because
of chance rather than an actual effect.


## Two-Way ANOVA


The mean sugar share was analyzed using a two-way ANOVA model, using `snap_status` and `region` as the
categorical variables, along with their interaction term, using an ordinary least squares (OLS) model.

A Type II ANOVA table was then computed to evaluate the statistical significance of these effects.


In [12]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.ols("sugar_share ~ C(snap_status) + C(region) + C(snap_status):C(region)", data=hh_sugar_df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

anova_table

,sum_sq,df,F,PR(>F)
C(snap_status),0.000145,1.0,0.035948,0.849658
C(region),0.045037,3.0,3.726256,0.011055
C(snap_status):C(region),0.025168,3.0,2.082288,0.100847
Residual,4.463952,1108.0,NaN,NaN


### Result

The two-way ANOVA shows that **SNAP status** has `F = 0.036` and `p = 0.8497`, **region** has `F = 3.726` and
`p = 0.0111`, and the **interaction** has `F = 2.082` and `p = 0.1008`. Since only the p-value for region is less than
`α = 0.05`, there is sufficient evidence to reject the null hypothesis for region but not for SNAP status and the
interaction.

This means that **only region has a significant effect** on the outcome. SNAP status does not show a meaningful
difference, and there is no evidence that its effect changes across regions.


## Chi-Square Test of Independence


The contingency table was constructed using `snap_status` and `high_sugar_status` to summarize the observed frequencies
of households across the categorical variables.


In [13]:
contingency = pd.crosstab(hh_transactions["snap_status"], hh_transactions["high_sugar_status"])

contingency

high_sugar_status,0,1
snap_status,,
0.0,37,69
1.0,304,706


The association between SNAP participation and high sugar status was evaluated using a chi-square test of independence,
comparing the observed counts with the expected counts under the assumption of independence. The expected frequencies
were also computed.


In [14]:
from scipy.stats import chi2_contingency

chi2, p_val, dof, expected = chi2_contingency(contingency)
expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)

display(expected_df)

print(f"x^2-stat = {chi2:.4f}")
print(f"p-value  = {p_val:.4f}")
print(f"d f      = {dof}")

high_sugar_status,0,1
snap_status,,
0.0,32.388889,73.611111
1.0,308.611111,701.388889


x^2-stat = 0.8303
p-value  = 0.3622
d f      = 1


### Result

The chi-square test of independence gave `χ² = 0.8303` and `p = 0.3622`. Since the p-value is greater than `α = 0.05`,
there is insufficient evidence to reject the null hypothesis.

This means there is no significant association between SNAP status and high sugar status. Any observed differences in
the counts are likely because of chance rather than an actual relationship.
